# Query Translation
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way so as to improve retrival.

**Semantic search on embeddings is hard to get right**. Embedding long documents is especially challenging. User queries are a challenge too. If the user provides an ambigious query, they'll end up get an ambiguous matches from embeddings and consequently an ambguous answer. The ambiguous matches land up in the LLM's context from which comes the LLM's response, which could lead to hallucinations.

In this notebook, we'll discuss the following techniques, including what each technique does and when to use them:
1. Multi Query
2. RAG Fusion
3. Query Decomposition
4. Step-back Prompting
5. HYDE (**HY**pothetical **D**ocument **E**mbedding)

><br/>
> **NOTE**: Each technique is covered in a separate, self-contained section of the notebook, so you will see code repeat in each section &amp; that is intentional.<br/><br/>

<br/>
<center>
<img src="images/rag_query_translation.png" width="600" height="480"/>
</center>

In this notebook, we will develop all these techniques on a vectorized content of a web-page. We'll be using the LangChain framework with **OpenAI GPT 5 Nano** LLM and **OpenAI Embeddings** in all examples here. You can replace these with any LLM & compatible embeddings of your choice - LangChain makes that really easy. **Note:** please use the same LLM and embedding combo across all techniques illustrated in this notebook. 

## 01 Multi-Query

**What is does**

* Takes the **user query** and asks the LLM to **generate several alternative** versions (_paraphrases/reformulations_) of that query.
* The goal is to **capture synonyms, different phrasings, and other angles** of the question.
* _Each generated query_ is sent to the vector store to fetch relevant context for that query. 
* Later, ALL contexts across ALL queries so fetched are merged together into a single context, which is fed to the LLM; RAG runs on the combined context.
* The intuition is that by asking the LLM the same question in N different ways, we will get more relevent chunks of data into the context, thereby improving overall response.

**Why use it?**

Different phrasings capture different embeddings → retrieve more relevant chunks. Helps reduce "embedding mismatch" (e.g., synonyms, domain-specific terms).

**Example:**

User asks: _"How do I cook pasta quickly?"_
LLM generates alternatives (3 variations in this case):
* "fast ways to prepare pasta"
* "quick pasta cooking methods"
* "rapid spaghetti preparation"

All queries are run against the embeddings  → retrieve docs covering microwaving, pressure cooker, etc. (there could be duplicates, so generate a unique list of retrievals). The retriever might otherwise miss some if only the original query was used.

**Key point:** Multi-query **improves recall** by broadening query formulations.

**Subtle difference**
* Multi-Query **does not** break the question into sub-questions.
* It simply **generates alternative rewrites** of the same question in its _entirity_.

The diagram below illustrates this technique.

<center>
<img src="images/rag_multi_query.png" width="700" height="300"/>
</center>

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [2]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [4]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)
# and OpenAI embeddings
embeddings = OpenAIEmbeddings()
chroma_store = pathlib.Path(os.getcwd()) / "chroma_db/chroma_index_rag_weng"

In [5]:
def create_or_load_embeddings(
    embeddings, chroma_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a ChromaDB embedding"""
    if not chroma_store.exists():
        # in this example we will load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        console.print(f"[yellow]Chunking the document. Please wait...[/yellow]")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()
        console.print(
            f"[yellow]Local embeddings created at {str(chroma_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(chroma_store)}[/yellow]"
        )
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()

    return retriever

In [6]:
retriever = create_or_load_embeddings(embeddings, chroma_store)

Loading document from URL https://lilianweng.github.io/posts/2023-06-23-agent/. Please wait...
Loaded 1 documents from URL
Metadata of first document: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
First 200 chars of first document: 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a 
Chunking the document. Please wait...
Created 214 chunks
Creating embeddings. Please wait...
Local embeddings created at c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_rag_advanced\chroma_db\chroma_index_rag_weng


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different formulations of same query
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

QUESTION = "What is task decomposition for LLM agents?"

In [8]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": QUESTION})

['How do large language model (LLM) agents perform task decomposition?',
 'How are tasks broken down into subtasks for LLM-powered agents?',
 'What techniques are used to decompose tasks in LLM agents?',
 'How does a complex goal get decomposed into steps an LLM agent can execute?',
 'What frameworks describe task decomposition for autonomous LLM-based agents?']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [9]:
from langchain_core.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [10]:
# Retrieve

# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": QUESTION})
print(f"Got {len(docs)} documents")
for i, doc in enumerate(docs):
    console.print(
        Markdown(f"### Document {i+1}\n{doc.page_content[:50] + "..."}\n---\n")
    )

# and print the retrival chain too
console.print(f"Retrieval chain: {retrieval_chain}")

Got 9 documents
Document 1

Subgoal and decomposition: The agent breaks down l...
Document 2

} ] Challenges# After going through key ideas and ...
Document 3

Another quite distinct approach, LLM+P (Liu et al....
Document 4

Component One: Planning# A complicated task usuall...
Document 5

The system comprises of 4 stages: (1) Task plannin...
Document 6

Boiko et al. (2023) also looked into LLM-empowered...
Document 7

Overview of a LLM-powered autonomous agent system....
Document 8

Challenges in long-term planning and task decompos...
Document 9

Task decomposition can be done (1) by LLM with sim...
Retrieval chain: first=ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language model assistant. Your task is to generate five \ndifferent versions of the given user question to retrieve relevant docu

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\2665704351.py:11: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\2665704351.py:11: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [11]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

multi_query_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = multi_query_rag_chain.invoke({"question": QUESTION})
console.print(f"[blue]Question:[/blue] {QUESTION}\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\2665704351.py:11: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


Question: What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the process of breaking a large, complex task into smaller, manageable subgoals or subtasks so the agent can handle it step by step. In practice:

 • The planning stage uses the LLM as the “brain” to parse the user request into multiple tasks, each with attributes li
 • Decomposition can be done in several ways:                                                                           
    • The LLM generates steps via simple prompts (e.g., “Steps for XYZ”).                                               
    • Task-specific instructions (e.g., “Write a story outline.”).                                                      
    • Guidance from human input.                                                                                        
 • The resulting subgoals and their dependencies enable scheduling and assignment to appropriate expert models.         


## 02 RAG Fusion
(Think of it as **Multi-Query + ranking / scoring**)

**What is it?**

A **retrieval re-ranking technique** inspired by "Reciprocal Rank Fusion" (RRF) in information retrieval ([see this blog for more on RRF](https://medium.com/@devalshah1619/mathematical-intuition-behind-reciprocal-rank-fusion-rrf-explained-in-2-mins-002df0cc5e2a))
* Generates multiple alternative queries (as we did in Multi-query technique) → retrieves documents for each query (multiple retrievals) → uses an algorithm (e.g., Reciprocal Rank Fusion) to rank documents (or contexts) so retrieved more intelligently
* You then **fuse/combine the rankes lists/results** from all queries using statistical fusion into one final ranking. This is _**not** plain concatenation_ (as you did in Multi-query).

**Why it's better than Multi-Query**
* **Multi-Query**: retrieve from multiple queries → concatenate.
* **RAG Fusion**: retrieve → rank contexts → select _only_ the best contexts based on RRF algo.

**Key idea**
* **Multiple phrasings + smart ranking → high recall AND high precision**.

**Subtle difference with Multi-Query**
* Both do query expansion (or paraphrasing), 
* But **RAG Fusion adds mathematical ranking, avoiding irrelevant noise**.

**How it works:**
* Each retrieval returns a ranked list (doc A rank=1, doc B rank=2, etc).
* Fusion scores docs by combining their **reciprocal ranks**:

$$
score(d) = \sum_{retrievers} \frac{1}{k+rank(d)}
$$
(with `k`= smoothening constant)
* Documents (more precisely, _contexts_) that appear across multiple queries/retrievers rise to the top.
* Reduces noise because only documents consistently relevant get boosted.

**Key Point:** RAG Fusion improves precision and robustness by rewarding cross-query/retriever consensus.

The diagram below illustrates this technique.

<center>
<img src="images/rag_fusion.png" width="700" height="300" alt="RAG Fusion"/>
</center>

Below is the code to implement **RAG Fusion**.

In [12]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [13]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [14]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)
# and OpenAI embeddings
embeddings = OpenAIEmbeddings()
chroma_store = pathlib.Path(os.getcwd()) / "chroma_db/chroma_index_rag_weng"

In [15]:
def create_or_load_embeddings(
    embeddings, chroma_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a ChromaDB embedding"""
    if not chroma_store.exists():
        # in this example we will load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        console.print(f"[yellow]Chunking the document. Please wait...[/yellow]")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()
        console.print(
            f"[yellow]Local embeddings created at {str(chroma_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(chroma_store)}[/yellow]"
        )
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()

    return retriever

In [16]:
retriever = create_or_load_embeddings(embeddings, chroma_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_rag_advanced\chroma_db\chroma_index_rag_weng


C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\699234573.py:51: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# RAG-Fusion: prompt
template = """You are a helpful assistant that generates multiple search queries 
based on a single input query.\n 
Return just a simple list of {num_queries} queries with no additional markup or text\n
Generate multiple search queries related to: {question} \n
Output ({num_queries} queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

QUESTION = "What is task decomposition for LLM agents?"

In [18]:
generate_queries = (
    prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke(
    {
        "num_queries": 5,
        "question": QUESTION,
    }
)

['What is task decomposition in the context of LLM agents?',
 'How can task decomposition improve planning in large language model agents?',
 'What are common strategies for decomposing tasks for LLM-driven agents?',
 'Can you provide examples of task decomposition for LLM agents in complex workflows?',
 'What are best practices and pitfalls in designing task decomposition for LLM agents?']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database. As a slight variation, this time the 5 is a parameter we pass to the prompt.

In [19]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """Reciprocal_rank_fusion that takes multiple lists of ranked documents
    and an optional parameter k used in the RRF formula"""

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key
            # (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [20]:
# Retrieve
from pydoc import doc

# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain_rf = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rf.invoke({"question": QUESTION, "num_queries": 5})
print(f"Got {len(docs)} documents")
# for i in range(5):
#     print(f"{docs[i]}\n\n")

# Note: docs -> List[(document, score)] (i.e. a list of tuples of (Document, score))
for i, (doc, score) in enumerate(docs[: len(docs) // 2]):
    print(f"Document #{i+1}\nContent: {doc.page_content}\nScore: {score}")

# # and print the retrival chain too
# console.print(f"Retrieval chain: {retrieval_chain}")

Got 8 documents
Document #1
Content: Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Score: 0.08278688524590164
Document #2
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Score: 0.06374807987711213
Document #3
Content: The system comprises of 4 stages:
(1) Task planning: LLM works as the brain and parses the user requests into multiple tasks. There are four attributes associated with each task: task type, ID, dependencies, and arguments. They use few-shot examples to guide LLM to do task parsing and planning.
Score: 0.049189141547682
Document #4
Content: Overview of a LLM-powered autonomous agent system.
Score: 0.048924731182795694


C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\2279208715.py:28: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [21]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain_rag_fusion = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain_rag_fusion.invoke({"question": QUESTION, "num_queries": 5})
console.print(f"[blue]Question:[/blue] {QUESTION}\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_8196\2665704351.py:11: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


Question: What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the process of breaking a large, complex user request into smaller, manageable subgoals or subtasks. This enables planning and execution in a structured way.

Key points:

 • It is part of the planning stage, where the LLM parses the request into multiple tasks, each with attributes such as 
 • It helps the agent manage dependencies and sequence tasks effectively.                                               
 • Decomposition can be done in several ways:                                                                           
    • By simple prompts to the LLM (e.g., “Steps for XYZ … 1., 2., 3.” or “What are the subgoals for achieving XYZ?”)   
    • By task-specific instructions (e.g., “Write a story outline”)                                                     
    • With human inputs to define subgoals or structure                                                                

## 03 Query Decomposition

**What is It?**

**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It's usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    _"Decompose this question into a list of simpler search queries."_
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

**Example**

User asks:

`Compare net profit trends and regulatory risks of Tesla over last 3 years.`

Decomposition may generate:
* "What are the net profit trends of Tesla in last 3 years?"
* "What are the regulatory risks faced by Tesla?"
* "How to compare these two aspects?"

**Key idea**
Break big task into smaller tasks → execute smaller tasks → retrieve for each smaller task → combine results.

**Subtle difference**
* **Multi-Query**: _same_ question phrased differently
* **Decomposition**: break question into _different sub-questions_ covering _different aspects_.

**Why / When to Use Query Decomposition**

Use it when:
* **Complex / multi-aspect questions:** 
    e.g., "Compare AutoGPT and BabyAGI, and explain how planning differs from memory."

* **Broad tasks spanning sub-topics:**
    e.g., "Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion."

* **Long, natural language queries:** with multiple clauses joined by "and", "or", "how" and also "¦".

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

Less helpful when:
* The query is **short and atomic** (e.g., "What is RAG Fusion?").
* The corpus is tiny or each document already covers the entire topic.

Once the query is broken decomposed into individual queries, there are two broad techniques to retrieve responses:
* Answer individually
* Answer recursively

We'll cover decomposition & the two answering techniques in this section. 

In [22]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [23]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [24]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)
# and OpenAI embeddings
embeddings = OpenAIEmbeddings()
chroma_store = pathlib.Path(os.getcwd()) / "chroma_db/chroma_index_rag_weng"

In [25]:
def create_or_load_embeddings(
    embeddings, chroma_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a ChromaDB embedding"""
    if not chroma_store.exists():
        # in this example we will load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        console.print(f"[yellow]Chunking the document. Please wait...[/yellow]")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()
        console.print(
            f"[yellow]Local embeddings created at {str(chroma_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(chroma_store)}[/yellow]"
        )
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()

    return retriever

In [26]:
retriever = create_or_load_embeddings(embeddings, chroma_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_rag_advanced\chroma_db\chroma_index_rag_weng


As a first step, let us ask the LLM our question & check it's response without any query decomposition.

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [28]:
QUESTION = "What is task decomposition for LLM agents?"

In [29]:
# first let's try WITHOUT decomposition
template = ChatPromptTemplate.from_template(
    "Answer the following question:\n\n{question}"
)
simple_chain = template | llm | StrOutputParser()
response = simple_chain.invoke({"question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the practice of breaking a complex goal into smaller, well-defined subtasks that the agent can handle in sequence (or in parallel). Each subtask has a clear input/output contract and, together, they progressively produce the final result. This approach makes reasoning more reliable, enables reuse of subresults, and helps manage tools, memory, and failures.

Key ideas

 • Hierarchical planning: Start with a high-level objective and recursively split it into finer subgoals until each piec
 • Orchestration: Decide the order, dependencies, and opportunities for parallel execution among subtasks.              
 • Interfaces: Define what each subtask expects as input and what it should deliver as output.                          
 • Context management: Pass relevant information between subtasks and maintain state across steps.                      
 • Tool usage: Allocate subtasks to the LLM’s reasoning, to external tools (search, calculators, APIs), or t

Now let's ask LLM to decompose our query as we discussed above.

In [30]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answered in isolation. \n
Generate multiple ({num_queries}) search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or additional quotes around the queries or markdown text \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [31]:
queries_generator = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
questions = queries_generator.invoke(
    {
        "num_queries": 5,
        "question": QUESTION,
    }
)
console.print(questions)

[
    'What is task decomposition in the context of LLM agents and why is it used',
    'Techniques for decomposing complex tasks for large language model agents',
    'Hierarchical task planning and decomposition for LLM agents',
    'Examples and frameworks of task decomposition for LLM-based agents',
    'Challenges, limitations, and evaluation of task decomposition in LLM agents'
]


## Answering Techniques for Query Decomposition
So you notice that we generated 5 different sub-questions related to an input questiont to improve our matches against the vector database.

Once we have the decomposed questions, we can use one of the following techniques to retrieve the results (&amp; the final response).

### 3a. Answering Individually

**How it works**:
* Break the big question into smaller sub-questions. 
* Retrieve documents **for each sub-question separately**. 
* Answer each sub-question independently using RAG.
* **Combine all sub-answers** into one final answer.

**Example**

_User asks_: `Compare AI regulations in the US, EU, and China.`

_Possible Sub-questions generated_:

* "What are AI regulations in the US?"
* "What are AI regulations in the EU?"
* "What are AI regulations in China?"

_Process_:
* Retrieve for (1), answer (1)
* Retrieve for (2), answer (2)
* Retrieve for (3), answer (3)
* Then combine.

**Key Properties**
* Independent answers → high recall
* More context diversity
* Less chance of missing a sub-topic
* More expensive → many retrieval + LLM calls == more 💸

**Subtle difference with _Anwer Recursively_** (which we will cover later)
* This method **does not use _intermediate answers_** to inform later ones.
* Each _sub-question is handled as if it is unrelated to the others_.

The image below illustrates this process visually:

![Multi Query](images/ans_individually.png)

In [34]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved 
    context to answer the question. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.\n
    Question: {question} \n
    Context: {context} \n
    Answer:"""
)

In [35]:
def retrieve_individual_qna(
    question, prompt_rag, sub_question_generator_chain, num_queries=5
):
    # Use our decomposition /
    sub_questions = sub_question_generator_chain.invoke(
        {"question": question, "num_queries": num_queries}
    )

    # Initialize a list to hold RAG chain results
    rag_results = []

    for sub_question in sub_questions:
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.invoke(sub_question)
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)

    return sub_questions, rag_results


def format_qna_pairs(questions, answers):
    """Format Q and A pairs"""

    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

In [36]:
questions, answers = retrieve_individual_qna(QUESTION, rag_prompt, queries_generator)

In [37]:
qna_pairs = format_qna_pairs(questions, answers)
print(qna_pairs)

Question 1: what is task decomposition for LLM agents
Answer 1: Task decomposition is the process of breaking a complex task into smaller, manageable subgoals so the agent can plan and execute steps more efficiently. In LLM agents, this can be done by simple prompts that elicit subgoals, by using task-specific instructions, or by incorporating human inputs. It is a planning step where the system parses requests into tasks with attributes like type, ID, dependencies, and arguments.

Question 2: how to decompose tasks for large language model agents
Answer 2: - Break down large tasks into smaller subgoals and plan ahead to manage complexity. 
- Decomposition can be done by prompting the LLM for steps or subgoals, using task-specific instructions, or incorporating human inputs. 
- Organize tasks with IDs and a dep field that encodes dependencies on prior task outputs, so downstream tasks can reuse resources like text, images, audio, or video.

Question 3: subtask planning for LLM-based ag

Now we feed each question & it's anwwer (i.e. complete output of previoius cell) as a context to LLM and ask it to extract answer from this context.

In [39]:
# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to generate an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = prompt | llm | StrOutputParser()

response = final_rag_chain.invoke({"context": qna_pairs, "question": QUESTION})

console.print(f"[blue]Question: [/blue] {QUESTION}\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

Question:  What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the planning step where a complex user request is broken into smaller, manageable subgoals or tasks. Each subtask is defined with attributes such as:

 • Type (what kind of task it is)                                                                                       
 • ID (a unique identifier)                                                                                             
 • Dependencies (which prior outputs it relies on)                                                                      
 • Arguments (parameters or inputs it needs)                                                                            

How it’s done:

 • LLM prompting to generate steps or subgoals                                                                          
 • Task-specific instructions tailored to the domain                                                                    
 

#### 3b. Answering Recursively
In this technique, the questions list we got above is passed recursively to the LLM - first $Q_1$ is passed and we get a response $A_1$ from LLM. $Q_1$ + $A_1$ is added as a context to $Q_2$ to get $A_2$, then ($Q_1$ + $A_1$) and ($Q_2$ + $A_2$) is added as a context when passing $Q_3$ to the LLM and so on. 

Finally, we land up with context -> {($Q_1$ + $A_1$), ($Q_2$ + $A_2$), ..., ($Q_{N-1}$ + $A_{N-1}$) } when $Q_N$ is passed to the LLM. The answer from the LLM to $Q_N with the above combined context is the final response. The intutition is by passing this "combined context" derived from the recursive process helps the LLM give a more coherent response to original question.

**Example**

User asks: 
* "Explain how a blockchain works and why it is secure".

Sub-questions:
1. "How does a blockchain work?"
2. "Why is it secure?"
3. "Explain the connection between (1) and (2)."

**Process:**
* Ask $Q_1$ get $A_1$
* Ask $Q_2 with {($Q_1 + $A_1)} as context and get $A_2 
* Ask $Q_3 with {($Q_1 + $A_1), ($Q_2 + $A_2)} as context and get $A_3
* $A_3 is our final answer

**Key Properties**
* Answers accumulate → more reasoning continuity 
* Produces cohesive explanation
* Less duplication
* Less RAG calls (because previous answers seed later ones)

**Subtle difference**
* This method uses the **answer of each sub-step to inform the next**, creating a chain of reasoning, much like a teacher building one idea after another.

The image below illustrates this process visually:

![Multi Query](images/ans_recursively.png)

In [40]:
# Prompt
template = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [41]:
from operator import itemgetter


def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


q_a_pairs = ""
for i, q in enumerate(questions):
    rag_chain = (
        {
            # get the context by asking the retriver to retrieve it
            # based on the question
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            # for the first question, q_a_pairs will be ""
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # format my prompt with above parameters
        | decomposition_prompt
        # ask LLM for response to formatted decomposition prompt
        | llm
        # parse out text as answer
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    console.print(f"[yellow]Intermediate QA-Pair #{i+1} -> [/yellow]")
    console.print(Markdown(q_a_pairs))

Intermediate QA-Pair #1 -> 
------------------------------------------------------------------------------------------------------------------------

Question: what is task decomposition for LLM agents Answer: Task decomposition for LLM agents is the process of breaking a complex user request into smaller, more manageable subgoals or subtasks. The agent then plans and executes these subtasks in a logical order to complete the overall task.

Key points:

 • Purpose: makes it easier for the agent to reason, plan ahead, and handle large tasks efficiently by limiting scope at
 • How it’s done:                                                                                                       
    • The system often uses a planning phase where the LLM parses the user request into multiple tasks.                 
    • Each task is described with attributes such as: task type, an ID, dependencies (which tasks must finish first), an
    • Subgoals can be generated in several ways: simple pr

Notice how we keep adding a Q & A pair to the overall context. At the end of all the questions (Q&A pairs), we get the final answer from the LLM, which we will display below.

In [42]:
console.print(f"[yellow]Final answer:[/yellow]")
console.print(Markdown(answer))

Final answer:
modular task decomposition in LLM agents

Summary

 • Modular task decomposition is the practice of breaking a complex user request into a structured set of smaller, reusa

Core ideas and what to model

 • Subtask attributes (for each subtask):                                                                               
    • id: unique identifier (e.g., T1, T2)                                                                              
    • type: what kind of work (gather, summarize, draft_outline, write, QA, code, etc.)                                 
    • description: a human-readable goal                                                                                
    • dep (dependencies): IDs of prerequisite subtasks                                                                  
    • args: inputs needed (text, data, URLs, parameters)                                                                
    • deliverable: expected output artifact              

## 04 Step Back
A different approach, presented by Google, is _Step-Back Prompting_. It takes the opposoite approach, where it tries to ask a more abstract question. [The paper](https://arxiv.org/pdf/2310.06117.pdf) talks a lot about using few-shot prompting to produce what they call the _step-back_ (or more abstract) questions. The way it does it is to provide a number of examples of step-back questions, given the original question.

**Simple explanation**

Before answering directly, the LLM creates a more general version of the question → retrieves information at a high level → uses that to answer the original question.

**Mental model**

Zoom out before zooming in.

**Example**

User asks: _"How do I implement a Kafka-based event-driven architecture for claims processing?"_

Step-back questions (don't worry if they sound challenging. After all the LLM will generate these for us!):

* "What are the core principles and design patterns of event-driven architectures?"
* "What are the key architectural requirements and processing challenges in insurance claims workflows?" 
* "What are the trade-offs and best practices when using message brokers for high-reliability, stateful business processes?"

Best for:
* Extremely narrow or specific queries
* When the vector store lacks specific content
* Knowledge-heavy topics where general understanding helps answer specifics

Differences:
* Unlike Multi-Query, Step-Back broadens the query instead of paraphrasing.
* Unlike Decomposition, it doesnâ€™t split into parts but moves to higher abstraction.

![Step Back](../images/step_back.png)

In [44]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [45]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [46]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)
# and OpenAI embeddings
embeddings = OpenAIEmbeddings()
chroma_store = pathlib.Path(os.getcwd()) / "chroma_db/chroma_index_rag_weng"

In [47]:
def create_or_load_embeddings(
    embeddings, chroma_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a ChromaDB embedding"""
    if not chroma_store.exists():
        # in this example we will load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        console.print(f"[yellow]Chunking the document. Please wait...[/yellow]")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()
        console.print(
            f"[yellow]Local embeddings created at {str(chroma_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(chroma_store)}[/yellow]"
        )
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()

    return retriever

In [48]:
retriever = create_or_load_embeddings(embeddings, chroma_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_rag_advanced\chroma_db\chroma_index_rag_weng


Let's look a some _few-shot_ examples

In [49]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindels was born in what country?",
        "output": "what is Jan Sindels personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [50]:
generate_queries_step_back = prompt | llm | StrOutputParser()
QUESTION = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": QUESTION})

'how do you break down a complex task into smaller, manageable steps?'

In [52]:
# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke({"question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking a large, complex user request into smaller, manageable subgoals or tasks, arranged so that the agent can plan, fetch required inputs, and execute them step by step. This enables efficient handling of complex tasks by modularizing work and tracking how outputs are used by later steps.

Key points:

 • Purpose: Turn a big task into a sequence of smaller tasks that are easier to plan, execute, and debug.               
 • How it fits in: It is a core part of the planning stage. The LLM acts as the “brain” and parses the user request into
 • Task representation: Each subtask is described with fields such as:                                                  
    • task type                                                                                                         
    • id (a unique task identifier)                                                                                     
    • dep (dependencies: the IDs of

### HYDE

**What is it?**

HYDE is an interesting approach that takes advantage of a very simple idea. The basic RAG flow takes a question and embeds it; takes a document & embeds it and looks for similarity between an embeded document & the embedded question. However, the question & document are very dis-similar. A document can very large and complex - may come from _dense_ publications (such as PDFs) and other sources, whereas questions are usually short & terse and could be ill-worded from users.

The intuition behind HyDE is this: take the questions and map them into document space using a hypothetical document (or by generating a hypothetical document). The idea is shown visually in the diagram below. In principle, for certain cases, a hypothetics document is _closer_ to desired document you want to retrieve from the high dimension vector space of the embedding than the sparse raw input question. This is a means of translating raw questions into hypotheticsl documents, which are better suited for retrieval.
 
![HYDE](../images/hyde.png)

In [54]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

os.environ["USER_AGENT"] = "LangChain_AdvRagTech/1.0"

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# as with the Naive RAG notebook, we'll use OpenAI LLM and OpenAI Embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# and ChromaDB as our vector store
from langchain_community.vectorstores import Chroma

In [55]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [56]:
# create our LLM - we will be using OpenAI GPT 3.5 Turbo (fairly cheap)
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)
# and OpenAI embeddings
embeddings = OpenAIEmbeddings()
chroma_store = pathlib.Path(os.getcwd()) / "chroma_db/chroma_index_rag_weng"

In [57]:
def create_or_load_embeddings(
    embeddings, chroma_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a ChromaDB embedding"""
    if not chroma_store.exists():
        # in this example we will load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        console.print(f"[yellow]Chunking the document. Please wait...[/yellow]")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()
        console.print(
            f"[yellow]Local embeddings created at {str(chroma_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(chroma_store)}[/yellow]"
        )
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=str(chroma_store),
        )
        retriever = vector_store.as_retriever()

    return retriever

In [58]:
retriever = create_or_load_embeddings(embeddings, chroma_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_rag_advanced\chroma_db\chroma_index_rag_weng


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """Please write a scienticic paper passage to answer the following question:
Question: {question}
Passage: """

prompt_hyde = ChatPromptTemplate.from_template(prompt_template)

generate_docs_for_retrieval = prompt_hyde | llm | StrOutputParser()

QUESTION = "What is task decomposition for LLM Agents?"
generated_doc = generate_docs_for_retrieval.invoke({"question": QUESTION})
console.print(Markdown(generated_doc))

Task decomposition for LLM agents refers to the deliberate partitioning of a high-level objective into a structured set of smaller, more tractable subproblems that can be addressed by an LLM and its accompanying tool interfaces. In this paradigm, the agent does not attempt to solve the entire goal in one monolithic prompt; instead, it generates, executes, and revises a hierarchy or graph of subtasks, each with defined inputs, outputs, and success criteria. This approach leverages the strengths of LLMs (flexible natural language reasoning, contextual inference, and code/tool orchestration) while mitigating their weaknesses (limited context window, tendency toward drift or hallucination, and fragility in long chains of reasoning) by imposing modular boundaries and explicit interfaces.

Core concepts

 • Goal and subgoal structure: A complex objective G is broken into a set of subtasks {T1, T2, …, Tn}, where each Ti has
 • Interfaces and tools: Subtasks are executed via prompts to the LLM

So we have generated a hypothetical document, which hopefully maps close to relevant documents in the larger vector embedding space. Now we can use this document to do a similarity search

In [60]:
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})
print(retrieved_docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.'), Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.'), Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='The system comprises of 4 stages:\n(1) Task planning: LLM works as the brain and parses the user requests into multiple tasks. There are four attributes associated with each task: task type, ID, dependencies, and arguments. They use few-shot examples to guide LLM to do task parsing and planning.'

In [61]:
# build a context from generated docs for LLM to use to answer our original question
# build context
context = ""
for doc in retrieved_docs:
    context += doc.page_content + "\n\n"

And now you RAG from retrieved documents :)

In [62]:
template = """Answer the following question based on the provided context.

{context}

Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(template)

final_chain = prompt_template | llm | StrOutputParser()
final_response = final_chain.invoke({"context": context, "question": question})
console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(final_response))

Question: What is task decomposition for LLM Agents?

AI Response:
Task decomposition for LLM Agents is the process of breaking a large, complex user request into a set of smaller, manageable subgoals or tasks. This enables the agent to plan, execute, and track progress more efficiently.

Key points:

 • Implemented in the Task Planning stage (Stage 1) where the LLM parses requests into multiple tasks.                  
 • Each subtask is described with four attributes: task type, ID, dependencies, and arguments.                          
 • Decomposition can be done by:                                                                                        
    • Simple LLM prompts (e.g., "Steps for XYZ" or "What are the subgoals for achieving XYZ?")                          
    • Task-specific instructions (e.g., "Write a story outline." for a writing task)                                    
    • Human input (optional guidance or constraints)                                        

### Summary

Above, we saw various techniques to transform user-queries at the head of the RAG pipeline. The table below summarizes all these approaches for quick reference:

| Technique | Use When | Avoid When |
|:----------|:---------|:-----------|
| Multi-Query | User query is vague/ambiguous. Terminology varies across documents. | If query is already precise. |
| RAG Fusion | You want highest accuracy. There are multiple retrieval signals. | Small datasets (not enough data to fuse).|
| Query Decomposition | Query is long, multi-part, or analytical. | Simple single-topic questions. |
| Step Back | Query is overly narrow, domain-specific, or uncommon. | When generalization hurts precision. |
| HYDE | Documents are sparse. Query is short/speculative. |  If vector store is dense and high-quality. | 

How to determine in which "class" a user's query falls at run-time is indeed an interesting challenge, which we will cover in a separate workbook.